In [4]:
import anthropic
from dotenv import load_dotenv
load_dotenv()

client = anthropic.Anthropic()

message = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=1024,
    messages=[{"role": "user", "content": "What is accounts receivable?"}]
)

print(message.content[0].text)
print(f"\nTokens used: {message.usage}")

# Accounts Receivable

**Accounts receivable (AR)** is money owed to a company by its customers for goods or services that have been delivered but not yet paid for.

## Key Points

- **It's an asset** – Listed on the balance sheet as a current asset
- **Represents credit sales** – The company has extended credit to customers
- **Expected to be collected** – Typically within 30, 60, or 90 days

## Simple Example

A business sells $1,000 worth of products to a customer on credit with "Net 30" payment terms. Until the customer pays:
- The $1,000 is recorded as accounts receivable
- Once paid, it becomes cash

## Why It Matters

- **Cash flow** – High AR means money is tied up waiting for payment
- **Business health** – Too much unpaid AR can indicate collection problems
- **Working capital** – Companies need to manage AR to maintain operations

The opposite concept is **accounts payable**, which is money a company owes to its suppliers.

Tokens used: Usage(cache_creation=CacheCreation(eph

In [5]:
from dotenv import load_dotenv
load_dotenv()

client = anthropic.Anthropic()

FINANCE_SYSTEM_PROMPT = """You are a senior finance analyst with deep expertise in IFRS 
accounting standards, multi-entity group reporting, and variance analysis.

Your outputs must:
- Use precise accounting terminology
- Quantify variances in both absolute ($) and percentage (%) terms  
- Flag any items requiring management attention
- Structure responses with clear headings
- Be concise — board members read these

When analysing data:
1. State the key finding first (headline number)
2. Explain the primary driver(s)
3. Note any anomalies or risks
4. Suggest one action item

Output format: plain text, no markdown except for line breaks."""

def finance_query(question: str, context: str = "") -> str:
    """Send a finance question with optional data context."""
    user_msg = question
    if context:
        user_msg = f"Context data:\n{context}\n\nQuestion: {question}"
    
    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=1024,
        temperature=0.1,         # low temp for consistent finance analysis
        system=FINANCE_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_msg}]
    )
    return response.content[0].text

# Test it with actual P&L data
pnl_context = """
EntityA December 2025:
- Revenue: $462,000 (Nov: $385,000, Budget: $390,000)
- COGS: $220,000 (Nov: $198,000)
- Gross Profit: $242,000 (52.4% margin)
- OpEx: $118,600 (Nov: $116,000, Budget: $120,000)
- Net Profit: $123,400 (26.7% margin)
"""

result = finance_query(
    question="Write a one-paragraph variance commentary on this P&L for the board pack.",
    context=pnl_context
)
print(result)

VARIANCE COMMENTARY — ENTITYA DECEMBER 2025

EntityA delivered net profit of $123,400 (26.7% margin), representing a $72,000 favourable variance to budget driven primarily by revenue outperformance of $72,000 (+18.5% vs budget), with December revenue of $462,000 also up $77,000 (+20.0%) month-on-month reflecting strong seasonal trading. Gross margin compressed 110bps to 52.4% (Nov: 53.5%) as COGS growth of 11.1% outpaced the revenue uplift on a proportional basis, suggesting either product mix shift or input cost pressure requiring review. OpEx was held $1,400 (1.2%) under budget at $118,600 despite the volume increase, indicating good cost discipline. Management should investigate the margin dilution to confirm whether this reflects a temporary December mix effect or an emerging trend requiring pricing action in Q1.


In [6]:
# A reusable library of finance prompts
# Add this to your GitHub portfolio

PROMPT_LIBRARY = {

    "variance_commentary": """Analyse the following P&L data and write a 
variance commentary suitable for a board pack. Structure: headline finding, 
primary drivers, risks, one recommended action. Max 150 words.
Data: {data}""",

    "gl_anomaly_describe": """Review this GL transaction and identify any anomalies:
unusual amounts, unexpected account pairings, round-number risk, 
duplicate patterns, or timing issues. Be specific.
Transaction: {transaction}""",

    "reconciliation_gap": """Explain this reconciliation gap in plain English 
for a non-finance manager. Then list the 3 most likely root causes ranked 
by probability. Include one question to ask the operations team.
Gap details: {gap_details}""",

    "ifrs_disclosure": """Draft an IFRS-compliant disclosure note for the 
following accounting item. Cite the relevant standard. Keep it concise 
but audit-defensible.
Item: {item}""",

    "cashflow_narrative": """Write a cash flow commentary for this period.
Highlight: operating cash conversion, investing activities, and any 
liquidity concerns. Include working capital movement analysis.
Data: {data}""",

    "budget_assumption_review": """Review these budget assumptions for 
reasonableness. Flag any that appear aggressive, conservative, or 
internally inconsistent. Suggest one sensitivity test per flagged item.
Assumptions: {assumptions}""",

    "month_end_checklist": """Generate a month-end close checklist for a 
{entity_type} with {num_entities} entities. Include: journal entries, 
reconciliations, intercompany eliminations, and statutory deadlines. 
Sequence by dependency.""",

    "intercompany_query": """Analyse this intercompany balance discrepancy.
Identify: likely cause, which entity needs to post a correcting entry, 
and whether this affects consolidated elimination. 
Details: {details}""",

    "aged_debt_commentary": """Write a collections commentary on this aged 
debtors report. Identify: top 3 accounts by risk, recommended provision 
movement, and suggested actions for the credit controller.
Report: {data}""",

    "audit_query_response": """Draft a response to this auditor query.
Be precise, cite documentation where relevant, and avoid language that 
could suggest uncertainty about the accounting treatment.
Query: {query}""",
}

def use_prompt(client, key: str, **kwargs) -> str:
    """Run a named prompt with the given variables."""
    prompt = PROMPT_LIBRARY[key].format(**kwargs)
    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=1024,
        temperature=0.1,
        system="You are a senior finance analyst specialising in IFRS reporting.",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

# Example usage
if __name__ == "__main__":
    import anthropic
    from dotenv import load_dotenv
    load_dotenv()
    client = anthropic.Anthropic()
    
    result = use_prompt(client, "variance_commentary",
        data="Revenue $462k vs $390k budget (+$72k, +18.5%). COGS $220k vs $185k budget (+$35k).")
    print(result)

## P&L Variance Commentary – Q[X] FY24

**Headline:** Revenue outperformed budget by $72k (+18.5%), delivering strong top-line growth, though gross margin compression partially offset gains.

**Primary Drivers:**
- Revenue uplift driven by [volume/pricing/mix – requires clarification from data owner]
- COGS overrun of $35k (+18.9%) exceeded revenue growth rate, indicating margin dilution
- Gross profit increased $37k in absolute terms, but gross margin declined from 52.6% (budget) to 52.4% (actual)

**Key Risk:**
COGS growth outpacing revenue suggests potential input cost inflation, supply chain inefficiencies, or unfavourable product mix that may accelerate if unaddressed.

**Recommended Action:**
Commission detailed COGS variance analysis by cost category and product line to identify margin leakage sources before next quarter's planning cycle.

---
*Prepared for Board review. Supporting schedules available upon request.*


In [7]:
import os

script = '''import os, anthropic
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()

FINANCE_SYSTEM_PROMPT = """You are a senior finance analyst with deep expertise in IFRS 
accounting standards, multi-entity group reporting, and variance analysis."""

PROMPT_LIBRARY = {
    "variance_commentary": "Analyse the following P&L data and write a board-ready variance commentary. Max 150 words. Data: {data}",
    "gl_anomaly_describe": "Review this GL transaction and identify anomalies: unusual amounts, round-number risk, duplicates, timing issues. Transaction: {transaction}",
    "reconciliation_gap": "Explain this reconciliation gap for a non-finance manager. List 3 likely root causes. Gap details: {gap_details}",
    "ifrs_disclosure": "Draft an IFRS-compliant disclosure note. Cite the relevant standard. Item: {item}",
    "aged_debt_commentary": "Write a collections commentary. Identify top 3 risk accounts and recommended actions. Report: {data}",
}

def use_prompt(key: str, **kwargs) -> str:
    prompt = PROMPT_LIBRARY[key].format(**kwargs)
    response = client.messages.create(
        model="claude-opus-4-5", max_tokens=1024, temperature=0.1,
        system=FINANCE_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

if __name__ == "__main__":
    print(use_prompt("variance_commentary",
        data="Revenue $462k vs $390k budget. COGS $220k. Net profit $123k (26.7%)"))
'''

os.makedirs("day2", exist_ok=True)
with open("day2/prompt_library.py", "w") as f:
    f.write(script)

print("Saved to day2/prompt_library.py")

Saved to day2/prompt_library.py
